# Batch PDF File-Based Extraction Evaluation

Evaluate PDF-based metadata extraction using OpenAI's native File API across open access Fuster validation samples.

**Approach:**
- Filter to open access PDFs only (via OpenAlex `is_oa` flag)
- Upload raw PDFs to OpenAI File API
- Use GPT-5-mini with native PDF understanding (text + visual analysis)
- Custom `PDF_SYSTEM_MESSAGE` optimized for document structure
- Compare against manually annotated ground truth
- Use `DatasetFeaturesNormalized` for ground truth validation

**Key Differences from Section-Based Pipeline:**
- No GROBID parsing required
- OpenAI processes both text AND images from each page
- Better for tables, figures, and visual content
- Higher token usage (text + image per page)

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# Project modules - using PDF file-based extraction with Prefect pipeline
from llm_metadata.gpt_extract import PDF_SYSTEM_MESSAGE
from llm_metadata.pdf_pipeline import (
    PDFClassificationConfig,
    PDFInputRecord,
    PDFOutputRecord,
    pdf_classification_flow,
    save_pdf_manifest,
)
from llm_metadata.openalex import get_work_by_doi, extract_pdf_url
from llm_metadata.schemas.fuster_features import (
    DatasetFeatures,
    DatasetFeaturesNormalized,
)
from llm_metadata.groundtruth_eval import (
    evaluate_indexed,
    EvaluationConfig,
    FuzzyMatchConfig,
    micro_average,
    macro_f1,
)

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Paths relative to project root
PDF_DIR = PROJECT_ROOT / "data/pdfs/fuster"
MANIFEST_PATH = PDF_DIR / "manifest.csv"
GROUND_TRUTH_PATH = PROJECT_ROOT / "data/dataset_092624.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "artifacts/pdf_file_results"

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF directory: {PDF_DIR}")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")
print(f"Ground truth exists: {GROUND_TRUTH_PATH.exists()}")

c:\Users\beav3503\AppData\Local\miniforge3\envs\llm\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'llm_metadata.schemas.abstract_metadata.DatasetAbstractMetadata'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\beav3503\AppData\Local\miniforge3\envs\llm\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'llm_metadata.schemas.abstract_metadata.DatasetAbstractMetadata'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Project root: c:\Users\beav3503\dev\llm_metadata
PDF directory: c:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster
Manifest exists: True
Ground truth exists: True


## Step 1: Load Manifest and Ground Truth

Map dataset DOIs to article DOIs and load ground truth annotations.

In [2]:
# Load manifest with DOI mapping
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest: {len(manifest_df)} total rows")

# Filter to downloaded PDFs only
downloaded_df = manifest_df[manifest_df['status'] == 'downloaded'].copy()
print(f"Downloaded PDFs: {len(downloaded_df)}")

# Create DOI mapping (dataset_doi -> article_doi)
doi_mapping = dict(zip(downloaded_df['dataset_doi'], downloaded_df['article_doi']))
print(f"DOI mappings: {len(doi_mapping)}")

# Display sample
downloaded_df[['article_doi', 'dataset_doi', 'downloaded_pdf_path']].head(5)

Manifest: 70 total rows
Downloaded PDFs: 67
DOI mappings: 67


,article_doi,dataset_doi,downloaded_pdf_path
0,10.1093/jhered/esx103,10.5061/dryad.121sk,data/pdfs/fuster/10.1093_jhered_esx103.pdf
1,10.1371/journal.pone.0128238,10.5061/dryad.1771t,data/pdfs/fuster/10.1371_journal.pone.0128238.pdf
2,10.1371/journal.pone.0073695,10.5061/dryad.1cc4v,data/pdfs/fuster/10.1371_journal.pone.0073695.pdf
3,10.1002/ece3.4685,10.5061/dryad.24q6q70,data/pdfs/fuster/10.1002_ece3.4685.pdf
4,10.1639/0007-2745-119.1.008,10.5061/dryad.24rj8,data/pdfs/fuster/10.1639_0007-2745-119.1.008.pdf


In [3]:
# Load ground truth
gt_df = pd.read_excel(GROUND_TRUTH_PATH)
print(f"Ground truth: {len(gt_df)} total rows")

# Extract dataset DOI from URL column
gt_df['dataset_doi'] = gt_df['url'].str.replace('https://doi.org/', '', regex=False)

# Filter to DOIs that have downloaded PDFs
gt_matched = gt_df[gt_df['dataset_doi'].isin(doi_mapping.keys())].copy()
print(f"Ground truth with matched PDFs: {len(gt_matched)}")

# Add article DOI mapping
gt_matched['article_doi'] = gt_matched['dataset_doi'].map(doi_mapping)

gt_matched[['dataset_doi', 'article_doi', 'data_type', 'species']].head(5)

Ground truth: 418 total rows
Ground truth with matched PDFs: 67


,dataset_doi,article_doi,data_type,species
2,10.5061/dryad.121sk,10.1093/jhered/esx103,EBV genetic analysis,Glyptemys insculpta
4,10.5061/dryad.1771t,10.1371/journal.pone.0128238,density,"raccoons, striped skunks"
6,10.5061/dryad.1cc4v,10.1371/journal.pone.0073695,presence only,Rangifer tarandus caribou
7,10.5061/dryad.24q6q70,10.1002/ece3.4685,other,Rangifer tarandus caribou
8,10.5061/dryad.24rj8,10.1639/0007-2745-119.1.008,genetic analysis,Aspicilia bicensis


## Step 2: Filter to Open Access PDFs Only

Query OpenAlex to identify open access papers and get PDF URLs where available.

In [4]:
# Query OpenAlex for open access status
from time import sleep

openalex_data = []
print("Querying OpenAlex for open access status...")

for idx, row in gt_matched.iterrows():
    article_doi = row['article_doi']
    dataset_doi = row['dataset_doi']
    
    try:
        work = get_work_by_doi(article_doi)
        if work:
            is_oa = work.get('open_access', {}).get('is_oa', False)
            oa_status = work.get('open_access', {}).get('oa_status', 'unknown')
            pdf_url = extract_pdf_url(work)
            
            openalex_data.append({
                'dataset_doi': dataset_doi,
                'article_doi': article_doi,
                'is_oa': is_oa,
                'oa_status': oa_status,
                'pdf_url': pdf_url,
            })
        else:
            openalex_data.append({
                'dataset_doi': dataset_doi,
                'article_doi': article_doi,
                'is_oa': False,
                'oa_status': 'not_found',
                'pdf_url': None,
            })
    except Exception as e:
        print(f"Error querying {article_doi}: {e}")
        openalex_data.append({
            'dataset_doi': dataset_doi,
            'article_doi': article_doi,
            'is_oa': False,
            'oa_status': 'error',
            'pdf_url': None,
        })
    
    # Rate limit to avoid hitting OpenAlex
    sleep(0.1)

openalex_df = pd.DataFrame(openalex_data)
print(f"\nOpenAlex query complete: {len(openalex_df)} records")

Querying OpenAlex for open access status...

OpenAlex query complete: 67 records


In [5]:
# Filter to open access only
oa_df = openalex_df[openalex_df['is_oa'] == True].copy()
print(f"Open access papers: {len(oa_df)} out of {len(openalex_df)}")

# Show OA status breakdown
print("\nOpen Access Status Breakdown:")
print(openalex_df['oa_status'].value_counts())

# Show how many have direct PDF URLs
has_pdf_url = oa_df['pdf_url'].notna().sum()
print(f"\nOA papers with direct PDF URL: {has_pdf_url}")
print(f"OA papers requiring local file: {len(oa_df) - has_pdf_url}")

Open access papers: 44 out of 67

Open Access Status Breakdown:
oa_status
gold         25
bronze       17
closed       17
not_found     6
hybrid        1
green         1
Name: count, dtype: int64

OA papers with direct PDF URL: 40
OA papers requiring local file: 4


In [6]:
# Validate ground truth with DatasetFeaturesNormalized (OA papers only)
RELEVANT_COLS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset'
]

# Filter ground truth to OA papers only
oa_dataset_dois = set(oa_df['dataset_doi'])
gt_oa = gt_matched[gt_matched['dataset_doi'].isin(oa_dataset_dois)].copy()

ground_truth_by_dataset = {}
validation_errors = []

for _, row in gt_oa.iterrows():
    dataset_doi = row['dataset_doi']
    try:
        # Extract relevant columns and convert to dict
        row_dict = {col: row[col] for col in RELEVANT_COLS if col in row.index}
        
        # Validate using DatasetFeaturesNormalized
        validated = DatasetFeaturesNormalized.model_validate(row_dict)
        ground_truth_by_dataset[dataset_doi] = validated
    except Exception as e:
        validation_errors.append({'doi': dataset_doi, 'error': str(e)})

print(f"Successfully validated: {len(ground_truth_by_dataset)} records")
print(f"Validation errors: {len(validation_errors)} records")

if validation_errors:
    print("\nFirst 3 errors:")
    for err in validation_errors[:3]:
        print(f"  {err['doi']}: {err['error'][:100]}...")

Successfully validated: 44 records
Validation errors: 0 records


## Step 3: Configure PDF File Pipeline

Set up PDF file-based extraction with OpenAI's native PDF support and custom system prompt.

In [7]:
IMPROVED_SPECIES_PROMPT = """
You are EcodataGPT, a structured data extraction engine for scientific PDF analysis.

Goal: Extract biodiversity dataset features from the provided PDF document into the provided schema.

## PDF Document Analysis

This PDF has been processed to provide both:
1. **Extracted text** from each page for detailed content analysis
2. **Visual rendering** of each page for tables, figures, and layout understanding

Use BOTH the text and visual information to extract accurate metadata. Pay special attention to:
- **Tables**: Often contain structured data about sampling sites, species lists, temporal coverage
- **Figures/Maps**: May show study area extent, spatial distribution, geographic coordinates
- **Supplementary materials**: Sometimes contain detailed methods or data descriptions

## Document Structure Guidance

Scientific papers typically follow this structure - prioritize sections accordingly:

| Section | Primary Information |
|---------|-------------------|
| Abstract | Overview summary (use for context, prefer details from body) |
| Introduction | Study objectives, geographic scope |
| Methods/Materials | **PRIMARY SOURCE**: Sampling protocols, study sites, dates, species |
| Study Area | Geographic coordinates, spatial extent, region names |
| Data/Data Availability | Dataset descriptions, data types, access information |
| Results | Actual measurements, confirmed values |
| Supplementary | Detailed species lists, coordinate tables, extended methods |

## Field Extraction Rules

1. **data_type**: What kind of biodiversity data was collected?
   - Look for measurement descriptions in Methods section
   - "counted individuals" → abundance; "recorded presence" → presence-only
   - "DNA samples/sequences" → genetic_analysis; "GPS tracks" → tracking
   - Check tables for data structure clues

2. **geospatial_info_dataset**: How is location data represented?
   - GPS coordinates in tables → "sample"; named sites → "site"
   - Range maps in figures → "range"; country/region text → "administrative_units"
   - Check figure captions for map descriptions

3. **spatial_range_km2**: Total study area extent in square kilometers
   - Look for explicit area mentions ("500 km²", "10,000 hectares")
   - Check maps for scale bars; calculate from bounding coordinates
   - Convert units: 1 ha = 0.01 km²; 1 mi² = 2.59 km²

4. **temporal_range / temp_range_i / temp_range_f**: Data collection period
   - Look in Methods for date ranges ("sampled from 2005 to 2010")
   - Check figure captions for time series dates
   - temp_range_i = start year, temp_range_f = end year

5. **species**: Taxonomic entities studied — CRITICAL FIELD
   
   **IMPORTANT GUIDANCE FOR SPECIES EXTRACTION:**
   - Extract the PRIMARY FOCAL species of the study (the main organisms being studied)
   - Include scientific names EXACTLY as written (e.g., "Rangifer tarandus")
   - Include common/vernacular names EXACTLY as written (e.g., "caribou", "wood turtle")
   - If both scientific and common names are given together, extract them as they appear
     Example: "wood turtle (Glyptemys insculpta)" → extract as "wood turtle (Glyptemys insculpta)"
   - For subspecies, include the full trinomial (e.g., "Rangifer tarandus caribou")
   - Include taxonomic groups if studied (e.g., "12 mammal species", "ground-dwelling beetles")
   
   **WHAT TO EXTRACT:**
   ✓ Main study organisms (the species the paper is about)
   ✓ Scientific names in italics or Latin binomials
   ✓ Common names of focal species
   ✓ Taxonomic counts if given ("41 fish species")
   
   **WHAT NOT TO EXTRACT (False Positives to Avoid):**
   ✗ Predators, prey, or associated species that are NOT the main study focus
   ✗ Species mentioned only in literature review or discussion comparisons
   ✗ Plant species mentioned only as habitat unless the study is about plants
   ✗ Pathogens/parasites unless the study specifically investigates them
   ✗ Generic taxonomic terms like "wildlife", "mammals", "insects" (unless specifically counted)
   
   **EXAMPLES:**
   - Paper about caribou migration → Extract: "Rangifer tarandus" or "caribou"
   - Paper about tick genetics → Extract: "black-legged tick" or "Ixodes scapularis", NOT "white-footed mouse" (a host)
   - Paper about forest ecosystem → Extract specific tree species studied, NOT general "forest trees"
   - Paper about predator-prey dynamics → Extract BOTH predator AND prey as focal species

## Extraction Philosophy

- Only use information EXPLICITLY stated or shown in the PDF. Do NOT guess or infer.
- If a field is not clearly stated, set it to null (or empty list).
- Prefer conservative outputs over over-extraction.
- When text and figures contradict, prefer quantitative data from tables/figures.
- Cross-reference Methods text with Results tables for verification.
- Output must conform exactly to the schema (types, enums, lists).
""".strip()

print("Improved species prompt defined")
print(f"Prompt length: {len(IMPROVED_SPECIES_PROMPT)} characters")

Improved species prompt defined
Prompt length: 4862 characters


In [8]:
# Pipeline configuration using PDFClassificationConfig
config = PDFClassificationConfig(
    model="gpt-5-mini",
    reasoning={"effort": "low"},
    max_output_tokens=4096,
    text_format=DatasetFeatures,
    use_native_pdf=True,  # Use OpenAI File API for native PDF understanding
    system_message=PDF_SYSTEM_MESSAGE,
    cleanup_files=False,  # Keep files for debugging
    max_workers=5,
    output_dir=OUTPUT_DIR,
)

print("PDF File Pipeline Configuration:")
print(f"  Model: {config.model}")
print(f"  Reasoning: {config.reasoning}")
print(f"  Max output tokens: {config.max_output_tokens}")
print(f"  Extraction schema: {config.text_format.__name__}")
print(f"  Mode: {'Native PDF (OpenAI File API)' if config.use_native_pdf else 'Text extraction'}")
print(f"\nSystem Prompt Preview (first 300 chars):")
print(f"  {config.system_message[:300]}...")

PDF File Pipeline Configuration:
  Model: gpt-5-mini
  Reasoning: {'effort': 'low'}
  Max output tokens: 4096
  Extraction schema: DatasetFeatures
  Mode: Native PDF (OpenAI File API)

System Prompt Preview (first 300 chars):
  You are EcodataGPT, a structured data extraction engine for scientific PDF analysis.

Goal: Extract biodiversity dataset features from the provided PDF document into the provided schema.

## PDF Document Analysis

This PDF has been processed to provide both:
1. **Extracted text** from each page for ...


## Step 4: Run PDF File-Based Extraction

Process all open access PDFs through OpenAI's File API.

In [9]:
# Prepare records for processing using PDFInputRecord
# Merge OA data with manifest to get PDF paths
oa_with_paths = oa_df.merge(
    downloaded_df[['dataset_doi', 'downloaded_pdf_path']],
    on='dataset_doi',
    how='left'
)

# Build PDFInputRecord list for Prefect pipeline
input_records = []
for _, row in oa_with_paths.iterrows():
    dataset_doi = row['dataset_doi']
    pdf_path_rel = row['downloaded_pdf_path']
    
    # Construct full PDF path
    pdf_path = PDF_DIR / pdf_path_rel if pdf_path_rel else None
    if pdf_path and not pdf_path.exists():
        pdf_path = PDF_DIR.parent / pdf_path_rel
    
    if pdf_path and pdf_path.exists():
        input_records.append(PDFInputRecord(
            id=dataset_doi,  # Use dataset DOI as unique identifier
            pdf_path=str(pdf_path),
            metadata={
                'article_doi': row['article_doi'],
                'dataset_doi': dataset_doi,
            }
        ))

print(f"PDFInputRecords created: {len(input_records)}")
print(f"All records will use local PDF files (native PDF mode)")

PDFInputRecords created: 0
All records will use local PDF files (native PDF mode)


In [10]:
# Run extraction using Prefect pdf_classification_flow
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_manifest_path = OUTPUT_DIR / f"pdf_file_results_{timestamp}.csv"

print(f"Running PDF file-based extraction on {len(input_records)} papers...")
print(f"Output manifest: {output_manifest_path}")
print()

# Run the Prefect flow
results = pdf_classification_flow(
    input_records=input_records,
    config=config,
    output_manifest=output_manifest_path,
)

# Summary
success_count = sum(1 for r in results if r.status == "success")
error_count = sum(1 for r in results if r.status == "error")
print(f"\nProcessing complete: {success_count} success, {error_count} errors")

Running PDF file-based extraction on 0 papers...
Output manifest: c:\Users\beav3503\dev\llm_metadata\artifacts\pdf_file_results\pdf_file_results_20260217_072133.csv



07:21:38.848 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8726
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

09:52:41.649 | INFO    | Flow run 'sly-dinosaur' - Beginning flow run 'sly-dinosaur' for flow 'pdf-classification-flow'

Processing 0 PDFs using native PDF (OpenAI File API) mode...
  Model: gpt-5-mini

Completed: 0 success, 0 failed
Total cost: $0.0000
Saved output manifest to c:\Users\beav3503\dev\llm_metadata\artifacts\pdf_file_results\pdf_file_results_20260217_072133.csv (0 records)


09:52:41.699 | INFO    | Flow run 'sly-dinosaur' - Finished in state Completed()


Processing complete: 0 success, 0 errors


In [11]:
# Results summary (manifest already saved by flow)
print(f"Results saved to: {output_manifest_path}")

# Create results DataFrame for downstream analysis
results_df = pd.DataFrame([r.model_dump() for r in results])

# Show extraction method breakdown
print("\nExtraction method breakdown:")
print(results_df['extraction_method'].value_counts())

Results saved to: c:\Users\beav3503\dev\llm_metadata\artifacts\pdf_file_results\pdf_file_results_20260217_072133.csv

Extraction method breakdown:


KeyError: 'extraction_method'

## Step 5: Prepare Extractions for Evaluation

Convert extraction results to Pydantic models for evaluation.

In [12]:
# Build predictions by dataset DOI
predictions_by_dataset = {}

for result in results:
    if result.status != "success" or not result.extraction:
        continue
    
    # PDFOutputRecord uses 'id' as the identifier (which is the dataset_doi)
    dataset_doi = result.id
    if not dataset_doi:
        continue
    
    try:
        # Create DatasetFeatures from extraction dict
        pred = DatasetFeatures.model_validate(result.extraction)
        predictions_by_dataset[dataset_doi] = pred
    except Exception as e:
        print(f"Error validating prediction for {dataset_doi}: {e}")

print(f"Valid predictions: {len(predictions_by_dataset)}")

# Find common DOIs between predictions and ground truth
common_dois = set(predictions_by_dataset.keys()) & set(ground_truth_by_dataset.keys())
print(f"Common DOIs for evaluation: {len(common_dois)}")

Valid predictions: 38
Common DOIs for evaluation: 38


## Step 6: Evaluate PDF File-Based Extraction

Compare PDF file-based extractions against ground truth using evaluation framework.

In [13]:
# Evaluation fields
EVAL_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f', 'species'
]

# Evaluation configuration with fuzzy matching for species
eval_config = EvaluationConfig(
    casefold_strings=True,
    treat_lists_as_sets=True,
    enhanced_species_matching=True,
    enhanced_species_threshold=70
)

# Filter to common DOIs
true_by_id = {doi: ground_truth_by_dataset[doi] for doi in common_dois}
pred_by_id = {doi: predictions_by_dataset[doi] for doi in common_dois}

# Run evaluation
pdf_report = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=EVAL_FIELDS,
    config=eval_config,
)

print("PDF FILE-BASED Extraction Metrics:")
print("=" * 70)
display(pdf_report.metrics_to_pandas())

PDF FILE-BASED Extraction Metrics:


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,25,112,21,0,38,0.182482,0.543478,0.273224,0.657895,0.000000
1,geospatial_info_dataset,16,161,4,0,38,0.090395,0.800000,0.162437,0.421053,0.000000
2,spatial_range_km2,17,6,10,9,38,0.739130,0.629630,0.680000,0.684211,0.684211
6,species,47,165,14,0,38,0.221698,0.770492,0.344322,1.236842,0.131579
5,temp_range_f,27,7,6,4,38,0.794118,0.818182,0.805970,0.815789,0.815789
4,temp_range_i,29,5,4,4,38,0.852941,0.878788,0.865672,0.868421,0.868421
3,temporal_range,1,36,32,1,38,0.027027,0.030303,0.028571,0.052632,0.052632


In [14]:
# Aggregate metrics
pdf_micro = micro_average(pdf_report.field_metrics.values())
pdf_macro = macro_f1(pdf_report.field_metrics.values())

print("\nAggregate Metrics:")
print("=" * 50)
print(f"{'Metric':<25} {'Value':>15}")
print("-" * 50)
print(f"{'Micro Precision':<25} {pdf_micro.precision or 0:>15.3f}")
print(f"{'Micro Recall':<25} {pdf_micro.recall or 0:>15.3f}")
print(f"{'Micro F1':<25} {pdf_micro.f1 or 0:>15.3f}")
print(f"{'Macro F1':<25} {pdf_macro or 0:>15.3f}")
print(f"{'Records Evaluated':<25} {len(common_dois):>15}")
print("=" * 50)


Aggregate Metrics:
Metric                              Value
--------------------------------------------------
Micro Precision                     0.248
Micro Recall                        0.640
Micro F1                            0.357
Macro F1                            0.451
Records Evaluated                      38


## Step 7: Per-Field Analysis

In [15]:
# Per-field metrics for PDF file-based extraction
field_data = []

for field in EVAL_FIELDS:
    fm = pdf_report.field_metrics.get(field)
    if fm:
        field_data.append({
            'field': field,
            'precision': round(fm.precision or 0, 3),
            'recall': round(fm.recall or 0, 3),
            'f1': round(fm.f1 or 0, 3),
            'tp': fm.tp,
            'fp': fm.fp,
            'fn': fm.fn,
            'exact_match_rate': round(fm.exact_match_rate or 0, 3),
        })

field_df = pd.DataFrame(field_data)
print("Per-Field Metrics:")
display(field_df)

# Identify best and worst performing fields
if len(field_data) > 0:
    best = max(field_data, key=lambda x: x['f1'])
    worst = min(field_data, key=lambda x: x['f1'])
    print(f"\nBest performing field: {best['field']} (F1={best['f1']})")
    print(f"Worst performing field: {worst['field']} (F1={worst['f1']})")

Per-Field Metrics:


,field,precision,recall,f1,tp,fp,fn,exact_match_rate
0,data_type,0.182,0.543,0.273,25,112,21,0.000
1,geospatial_info_dataset,0.090,0.800,0.162,16,161,4,0.000
2,spatial_range_km2,0.739,0.630,0.680,17,6,10,0.684
3,temporal_range,0.027,0.030,0.029,1,36,32,0.053
4,temp_range_i,0.853,0.879,0.866,29,5,4,0.868
5,temp_range_f,0.794,0.818,0.806,27,7,6,0.816
6,species,0.222,0.770,0.344,47,165,14,0.132



Best performing field: temp_range_i (F1=0.866)
Worst performing field: temporal_range (F1=0.029)


In [16]:
# Detailed per-record comparison results
print("Per-record, per-field results (mismatches highlighted):")
results_detail_df = pdf_report.to_pandas()
mismatches = results_detail_df[~results_detail_df['match']]
print(f"\nTotal mismatches: {len(mismatches)}")
display(mismatches)

Per-record, per-field results (mismatches highlighted):

Total mismatches: 169


,record_id,field,true_value,pred_value,match,tp,fp,fn,tn
0,10.5061/dryad.121sk,data_type,[genetic_analysis],"[genetic_analysis, abundance, time_series]",False,1,2,0,0
1,10.5061/dryad.121sk,geospatial_info_dataset,None,"[site, administrative_units, site_ids]",False,0,3,0,0
3,10.5061/dryad.121sk,temporal_range,2006-2007,2006–2007,False,0,1,1,0
7,10.5061/dryad.1771t,data_type,[density],[abundance],False,0,1,1,0
8,10.5061/dryad.1771t,geospatial_info_dataset,[sample],"[sample, site, administrative_units, maps, sit...",False,1,4,0,0
...,...,...,...,...,...,...,...,...,...
255,10.5061/dryad.t11f5,temporal_range,2007,8 June 2007 to 15 August 2007,False,0,1,1,0
258,10.5061/dryad.t11f5,species,[10 fish species],"[White sucker Catostomus commersoni, Rock bass...",False,0,10,1,0
259,10.5061/dryad.xksn02vb9,data_type,[other],"[traits, genetic_analysis]",False,0,2,1,0
260,10.5061/dryad.xksn02vb9,geospatial_info_dataset,None,"[site, administrative_units, site_ids]",False,0,3,0,0


## Step 8: Cost Analysis

In [17]:
# Cost analysis from results
successful_results = [r for r in results if r.status == "success"]

if successful_results:
    total_cost = sum(r.cost_usd or 0 for r in successful_results)
    total_input_tokens = sum(r.input_tokens or 0 for r in successful_results)
    total_output_tokens = sum(r.output_tokens or 0 for r in successful_results)
    
    print("\nCOST ANALYSIS (PDF File-Based Extraction)")
    print("=" * 50)
    print(f"{'Metric':<30} {'Value':>18}")
    print("-" * 50)
    print(f"{'Total PDFs Processed':<30} {len(successful_results):>18}")
    print(f"{'Avg Input Tokens per PDF':<30} {total_input_tokens / len(successful_results):>18,.0f}")
    print(f"{'Avg Output Tokens per PDF':<30} {total_output_tokens / len(successful_results):>18,.0f}")
    print("-" * 50)
    print(f"{'Total Input Tokens':<30} {total_input_tokens:>18,}")
    print(f"{'Total Output Tokens':<30} {total_output_tokens:>18,}")
    print(f"{'Total Cost (USD)':<30} ${total_cost:>17.4f}")
    print(f"{'Avg Cost per PDF (USD)':<30} ${total_cost / len(successful_results):>17.5f}")
    print("=" * 50)
    
    # Method breakdown
    url_results = [r for r in successful_results if r.extraction_method == 'openai_url_api']
    file_results = [r for r in successful_results if r.extraction_method == 'openai_file_api']
    
    if url_results:
        url_cost = sum(r.cost_usd or 0 for r in url_results)
        print(f"\nURL-based extraction: {len(url_results)} papers, ${url_cost:.4f} total")
    
    if file_results:
        file_cost = sum(r.cost_usd or 0 for r in file_results)
        print(f"File upload extraction: {len(file_results)} papers, ${file_cost:.4f} total")


COST ANALYSIS (PDF File-Based Extraction)
Metric                                      Value
--------------------------------------------------
Total PDFs Processed                           38
Avg Input Tokens per PDF                   25,066
Avg Output Tokens per PDF                   1,856
--------------------------------------------------
Total Input Tokens                        952,500
Total Output Tokens                        70,512
Total Cost (USD)               $           0.4995
Avg Cost per PDF (USD)         $          0.01314
File upload extraction: 38 papers, $0.4995 total
